# Task 163: harmonica_gravity — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Gravity inversion using harmonica equivalent sources

**Paper**: Harmonica: Forward modeling, inversion, and processing gravity and magnetic data
**Repository**: https://github.com/fatiando/harmonica

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 45.45 dB
- **SSIM**: 0.9924
- **Evaluation**: Gravity field inversion (CC=0.9993, RMSE=0.0758 mGal)

### Mathematical Formulation

**Blind Source Separation**: observed mixtures $\mathbf{x}(t) = A\mathbf{s}(t)$

**ICA** finds unmixing $W$: $\hat{\mathbf{s}} = W\mathbf{x}$ maximizing non-Gaussianity:
$$\max_{\mathbf{w}} |E\{G(\mathbf{w}^T\mathbf{x})\} - E\{G(\nu)\}|$$

**NMF**: $V \approx WH$, with $W, H \geq 0$, minimizing $D(V\|WH)$.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
Task 163: harmonica_gravity — Gravity Field Inversion using Equivalent Sources

Inverse Problem: From surface gravity observations, infer subsurface density
distribution using equivalent sources (harmonica library).

Pipeline:
  1. Define subsurface density model (rectangular prisms)
  2. Forward: Compute gravity anomaly at surface using harmonica.prism_gravity
  3. Add Gaussian noise to simulate noisy observations
  4. Inverse: Fit harmonica.EquivalentSources to the noisy data
  5. Predict/reconstruct the gravity field from the equivalent source model
  6. Evaluate reconstruction quality (PSNR, SSIM, CC, RMSE)
  7. Save outputs and produce 4-panel visualization
"""

import os
import sys
import numpy as np
import matplotlib

## 3. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim_func, peak_signal_noise_ratio as psnr_func

import harmonica as hm
import verde as vd

# ──────────────────────────────────────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────────────────────────────────────
SANDBOX = "/data/yjh/harmonica_gravity_sandbox"
ASSETS  = "/data/yjh/website_assets/Task_163_harmonica_gravity"
os.makedirs(SANDBOX, exist_ok=True)
os.makedirs(ASSETS, exist_ok=True)

SEED = 42
np.random.seed(SEED)

# ──────────────────────────────────────────────────────────────────────────────

### Step 1: Define subsurface density model (rectangular prisms)

Intermediate processing step in the pipeline.

In [ ]:
# Step 1: Define subsurface density model (rectangular prisms)
# ──────────────────────────────────────────────────────────────────────────────
# Each prism: [west, east, south, north, bottom, top] in meters
# Positive density contrast = denser than surroundings
prisms = [
    [-3000, -500, -3000, -500, -5000, -1000],   # shallow dense block (NW)
    [1000, 4000, 1000, 4000, -8000, -3000],      # deeper block (SE)
    [-2000, 1000, 3000, 6000, -4000, -1500],     # medium block (NE)
    [4000, 7000, -6000, -3000, -6000, -2000],    # block (S)
]
densities = np.array([500.0, -400.0, 300.0, -250.0])  # kg/m³ anomalies

print(f"[INFO] Defined {len(prisms)} subsurface prisms")
for i, (p, d) in enumerate(zip(prisms, densities)):
    print(f"  Prism {i}: bounds={p}, density_contrast={d} kg/m³")

# ──────────────────────────────────────────────────────────────────────────────

### Step 2: Forward — compute gravity anomaly on a regular surface grid

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Step 2: Forward — compute gravity anomaly on a regular surface grid
# ──────────────────────────────────────────────────────────────────────────────
region = (-10000, 10000, -10000, 10000)   # meters (20 km × 20 km)
shape = (80, 80)
observation_height = 0.0  # meters (at surface)

# verde.grid_coordinates returns (easting, northing, ...) as 2D arrays
coordinates = vd.grid_coordinates(
    region, shape=shape, extra_coords=observation_height
)

print(f"[INFO] Observation grid: {shape[0]}×{shape[1]} points, region={region}")
print(f"[INFO] Observation height: {observation_height} m")

# Compute gravity using the analytical prism formula
# field="g_z" gives the vertical component in m/s² (or mGal depending on version)
gravity_true = hm.prism_gravity(
    coordinates=coordinates,
    prisms=prisms,
    density=densities,
    field="g_z",
)

# Convert to mGal if needed (harmonica returns SI = m/s²; 1 mGal = 1e-5 m/s²)
# Check magnitude to determine units
gravity_max = np.max(np.abs(gravity_true))
print(f"[INFO] True gravity range: [{gravity_true.min():.6e}, {gravity_true.max():.6e}]")

# If values are very small (SI units), convert to mGal for readability
if gravity_max < 1e-2:
    gravity_true_mgal = gravity_true * 1e5  # convert m/s² → mGal
    unit_label = "mGal"
    noise_level = 0.5  # mGal
    print(f"[INFO] Converted to mGal. Range: [{gravity_true_mgal.min():.4f}, {gravity_true_mgal.max():.4f}] mGal")
else:
    gravity_true_mgal = gravity_true
    unit_label = "mGal" if gravity_max < 100 else "m/s²"
    noise_level = 0.5
    print(f"[INFO] Values in native units. Range: [{gravity_true_mgal.min():.4f}, {gravity_true_mgal.max():.4f}] {unit_label}")

# ──────────────────────────────────────────────────────────────────────────────

### Step 3: Add noise to simulate observations

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Step 3: Add noise to simulate observations
# ──────────────────────────────────────────────────────────────────────────────
noise = np.random.normal(0, noise_level, gravity_true_mgal.shape)
gravity_noisy = gravity_true_mgal + noise
print(f"[INFO] Added Gaussian noise: σ = {noise_level} {unit_label}")
print(f"[INFO] Noisy gravity range: [{gravity_noisy.min():.4f}, {gravity_noisy.max():.4f}] {unit_label}")

# ──────────────────────────────────────────────────────────────────────────────

### Step 4: Inverse — fit EquivalentSources model

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Step 4: Inverse — fit EquivalentSources model
# ──────────────────────────────────────────────────────────────────────────────
print("[INFO] Fitting EquivalentSources model...")

# Flatten the coordinates for fitting (verde gives 2D arrays)
easting_flat = coordinates[0].ravel()
northing_flat = coordinates[1].ravel()
height_flat = np.full_like(easting_flat, observation_height)
data_flat = gravity_noisy.ravel()

fit_coords = (easting_flat, northing_flat, height_flat)

# EquivalentSources: depth controls source placement depth below data points
# damping controls regularization (Tikhonov)
eqs = hm.EquivalentSources(
    depth=5000,      # sources placed 5 km below observation points
    damping=1e-3,    # regularization parameter
)
eqs.fit(fit_coords, data_flat)
print("[INFO] EquivalentSources model fitted successfully")

# ──────────────────────────────────────────────────────────────────────────────

### Step 5: Predict / reconstruct gravity field

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Step 5: Predict / reconstruct gravity field
# ──────────────────────────────────────────────────────────────────────────────
predicted_flat = eqs.predict(fit_coords)
gravity_reconstructed = predicted_flat.reshape(shape)

print(f"[INFO] Reconstructed gravity range: [{gravity_reconstructed.min():.4f}, {gravity_reconstructed.max():.4f}] {unit_label}")

# Also predict on upward-continued surface (to demonstrate interpolation capability)
upward_height = 2000.0  # 2 km above surface
coords_up = vd.grid_coordinates(region, shape=shape, extra_coords=upward_height)
easting_up = coords_up[0].ravel()
northing_up = coords_up[1].ravel()
height_up = np.full_like(easting_up, upward_height)
predicted_upward = eqs.predict((easting_up, northing_up, height_up)).reshape(shape)
print(f"[INFO] Upward continued ({upward_height}m) range: [{predicted_upward.min():.4f}, {predicted_upward.max():.4f}] {unit_label}")

# ──────────────────────────────────────────────────────────────────────────────

### Step 6: Evaluation metrics

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# Step 6: Evaluation metrics
# ──────────────────────────────────────────────────────────────────────────────
# Use the true gravity field (in mGal) as ground truth
gt = gravity_true_mgal
recon = gravity_reconstructed
residual = gt - recon

# RMSE
rmse = np.sqrt(np.mean(residual**2))

# Correlation Coefficient
cc = np.corrcoef(gt.ravel(), recon.ravel())[0, 1]

# Normalize both to [0, 1] for PSNR/SSIM computation
gt_min, gt_max = gt.min(), gt.max()
data_range = gt_max - gt_min

if data_range > 0:
    gt_norm = (gt - gt_min) / data_range
    recon_norm = (recon - gt_min) / data_range
    recon_norm = np.clip(recon_norm, 0, 1)
else:
    gt_norm = np.zeros_like(gt)
    recon_norm = np.zeros_like(recon)

# PSNR
psnr_val = psnr_func(gt_norm, recon_norm, data_range=1.0)

# SSIM
ssim_val = ssim_func(gt_norm, recon_norm, data_range=1.0)

print(f"\n{'='*60}")
print(f"  EVALUATION METRICS")
print(f"{'='*60}")
print(f"  PSNR  = {psnr_val:.2f} dB")
print(f"  SSIM  = {ssim_val:.4f}")
print(f"  CC    = {cc:.6f}")
print(f"  RMSE  = {rmse:.4f} {unit_label}")
print(f"{'='*60}\n")

# ──────────────────────────────────────────────────────────────────────────────

### Step 7: Save outputs

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Step 7: Save outputs
# ──────────────────────────────────────────────────────────────────────────────
# gt_output.npy  = true gravity field (2D array)
# recon_output.npy = reconstructed gravity field (2D array)
np.save(os.path.join(SANDBOX, "gt_output.npy"), gt)
np.save(os.path.join(SANDBOX, "recon_output.npy"), recon)

# Also save to assets
np.save(os.path.join(ASSETS, "gt_output.npy"), gt)
np.save(os.path.join(ASSETS, "recon_output.npy"), recon)

# Save metrics
metrics = {
    "psnr_db": float(psnr_val),
    "ssim": float(ssim_val),
    "cc": float(cc),
    "rmse_mgal": float(rmse),
    "noise_level_mgal": float(noise_level),
    "n_prisms": len(prisms),
    "grid_shape": list(shape),
    "region_m": list(region),
}
import json
with open(os.path.join(SANDBOX, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
with open(os.path.join(ASSETS, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print("[INFO] Saved gt_output.npy, recon_output.npy, metrics.json")

# ──────────────────────────────────────────────────────────────────────────────

### Step 8: 4-panel visualization

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Step 8: 4-panel visualization
# ──────────────────────────────────────────────────────────────────────────────
easting_km = coordinates[0] / 1000.0
northing_km = coordinates[1] / 1000.0
extent = [easting_km.min(), easting_km.max(), northing_km.min(), northing_km.max()]

# Shared color limits for first 3 panels
vmin = min(gt.min(), gravity_noisy.min(), recon.min())
vmax = max(gt.max(), gravity_noisy.max(), recon.max())

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Panel 1: True gravity anomaly
ax = axes[0, 0]
im1 = ax.imshow(gt, extent=extent, origin="lower", cmap="RdBu_r", vmin=vmin, vmax=vmax)
ax.set_title("(a) True Gravity Anomaly", fontsize=13, fontweight="bold")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
plt.colorbar(im1, ax=ax, label=f"Gravity ({unit_label})", shrink=0.85)

# Panel 2: Noisy observations
ax = axes[0, 1]
im2 = ax.imshow(gravity_noisy, extent=extent, origin="lower", cmap="RdBu_r", vmin=vmin, vmax=vmax)
ax.set_title(f"(b) Noisy Observations (σ={noise_level} {unit_label})", fontsize=13, fontweight="bold")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
plt.colorbar(im2, ax=ax, label=f"Gravity ({unit_label})", shrink=0.85)

# Panel 3: Reconstructed (Equivalent Sources)
ax = axes[1, 0]
im3 = ax.imshow(recon, extent=extent, origin="lower", cmap="RdBu_r", vmin=vmin, vmax=vmax)
ax.set_title("(c) Equivalent Source Reconstruction", fontsize=13, fontweight="bold")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
plt.colorbar(im3, ax=ax, label=f"Gravity ({unit_label})", shrink=0.85)

# Panel 4: Residual
ax = axes[1, 1]
res_abs_max = max(abs(residual.min()), abs(residual.max()))
im4 = ax.imshow(residual, extent=extent, origin="lower", cmap="RdBu_r",
                vmin=-res_abs_max, vmax=res_abs_max)
ax.set_title("(d) Residual (True − Reconstructed)", fontsize=13, fontweight="bold")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
plt.colorbar(im4, ax=ax, label=f"Residual ({unit_label})", shrink=0.85)

fig.suptitle(
    f"Gravity Field Inversion via Equivalent Sources\n"
    f"PSNR={psnr_val:.1f} dB | SSIM={ssim_val:.4f} | CC={cc:.4f} | RMSE={rmse:.3f} {unit_label}",
    fontsize=14, fontweight="bold", y=1.02
)

plt.tight_layout()
vis_path_sandbox = os.path.join(SANDBOX, "vis_result.png")
vis_path_assets = os.path.join(ASSETS, "vis_result.png")
fig.savefig(vis_path_sandbox, dpi=150, bbox_inches="tight")
fig.savefig(vis_path_assets, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"[INFO] Saved visualization: {vis_path_sandbox}")
print(f"[INFO] Saved visualization: {vis_path_assets}")

# ──────────────────────────────────────────────────────────────────────────────
# Summary
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  TASK 163: harmonica_gravity — COMPLETE")
print(f"{'='*60}")
print(f"  Forward model: {len(prisms)} rectangular prisms")
print(f"  Grid: {shape[0]}×{shape[1]} @ {observation_height}m elevation")
print(f"  Inverse: EquivalentSources (depth=5000m, damping=1e-3)")
print(f"  PSNR  = {psnr_val:.2f} dB")
print(f"  SSIM  = {ssim_val:.4f}")
print(f"  CC    = {cc:.6f}")
print(f"  RMSE  = {rmse:.4f} {unit_label}")
print(f"{'='*60}")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

## 4. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **harmonica_gravity**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=45.45 dB, SSIM=0.9924)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Harmonica: Forward modeling, inversion, and processing gravity and magnetic data
- Repository: https://github.com/fatiando/harmonica
